In [8]:
import os
import pandas as pd

# Define the folder where the CSV files are located
folder_path = './'

# Initialize an empty list to store the data
data = []

# Loop through each file in the folder
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        
        # Read the CSV file into a DataFrame
        df = pd.read_csv(file_path)
        
        # Filter and group the data by SMT Depth, Proof System, and Curve
        grouped = df.groupby(['SMT Depth', 'Proof System', 'Curve'])
        
        for (smt_depth, proof_system, curve), group in grouped:
            # Calculate the average proof generation time and proof verification time
            avg_gen_time = group[group['Type'] == 'proof-generation']['Duration (ms)'].mean()
            avg_ver_time = group[group['Type'] == 'proof-verification']['Duration (ms)'].mean()
            num_constraints = len(group[group['Type'] == 'proof-generation'])  # Using count as constraints

            # Append the data to the list
            data.append({
                'Proof System': proof_system,
                'SMT Depth': smt_depth,
                'Number of Constraints': num_constraints,
                'Proof Generation Time (ms)': avg_gen_time,
                'Proof Verification Time (ms)': avg_ver_time,
                'Curve': curve
            })

# Create a DataFrame from the data
result_df = pd.DataFrame(data)

# Check if the pair (254, 'fflonk') exists
if not ((result_df['SMT Depth'] == 254) & (result_df['Proof System'] == 'fflonk')).any():
    # If it doesn't exist, add the row with NaN values for unspecified columns
    new_row = {'Proof System': 'fflonk', 'SMT Depth': 254, 
               'Number of Constraints': pd.NA, 
               'Proof Generation Time (ms)': pd.NA, 
               'Proof Verification Time (ms)': pd.NA, 
               'Curve': pd.NA}  # Optional Curve field if needed
    result_df = pd.concat([result_df, pd.DataFrame([new_row])], ignore_index=True)

# Define custom sort order for Proof System
proof_system_order = ['groth16', 'plonk', 'fflonk']

# Convert the 'Proof System' column to a categorical type with the specified order
result_df['Proof System'] = pd.Categorical(result_df['Proof System'], categories=proof_system_order, ordered=True)

# Sort by 'SMT Depth' and then by 'Proof System' with custom order
sorted_df = result_df.sort_values(by=['SMT Depth', 'Proof System'])

# Print the sorted DataFrame
print(sorted_df.to_string(index=False))


Proof System  SMT Depth Number of Constraints Proof Generation Time (ms) Proof Verification Time (ms) Curve
     groth16         10                    49                 595.895638                     238.5096 bn128
       plonk         10                    49               23155.911704                   239.083488 bn128
      fflonk         10                    49               32036.210133                   254.236632 bn128
     groth16        254                    49                4370.937712                   229.242474 bn128
       plonk        254                     3              814942.789944                   335.704792 bn128
      fflonk        254                  <NA>                       <NA>                         <NA>  <NA>
